# Baseline evaluation — FinVerify

This notebook runs a small, open-weights HuggingFace model (**Qwen2.5-3B-Instruct**) as the *baseline* financial advisor. It samples **5 questions from each of the three dataset subsets** (`standard_questions`, `open_ended_hard`, `reddit_questions`) and grades the answers with three signals:

1. **Multiple-choice accuracy** — exact letter match (only applies to MC items).
2. **Embedding similarity** — cosine similarity between the model answer and the reference `correct_answer`, using `BAAI/bge-small-en-v1.5`.
3. **LLM-as-judge rubric** — Claude scores the answer on `factuality`, `completeness`, and `advice_quality` (1–5 each).

Why a small open-weights baseline? The RAG notebook will use a frontier API model (Claude). Comparing Qwen-3B (no retrieval) against Claude + RAG would confound *retrieval gains* with *model-size gains*. The cleanest comparison is to also run **Claude with no retrieval** as a second baseline — we add that optionally at the bottom.

## 1. Setup

Install dependencies (uncomment the `%pip install` line on first run). Requires Python 3.10+.

The `ANTHROPIC_API_KEY` environment variable must be set for the LLM-judge step. Put it in your shell or a `.env` file; the notebook will skip the judge if it's missing and just show embedding-similarity scores.

In [3]:
# Dependency installs
!pip install -U pip setuptools wheel

!pip install \
  "numpy==1.24.4" \
  "scipy==1.10.1" \
  "scikit-learn==1.2.2" \
  "pillow>=10.0.0" \
  "torch>=2.2" \
  "transformers>=4.44" \
  "accelerate>=0.33" \
  "sentence-transformers>=3.0" \
  "chromadb>=0.5" \
  "requests>=2.32" \
  "pypdf>=4.0" \
  "beautifulsoup4>=4.12" \
  "anthropic>=0.34" \
  "datasets>=2.20" \
  "pandas>=2.2" \
  "jupyter>=1.0" \
  "python-dotenv>=1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.7 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 104.9 MB/s  0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'numpy' when getting requirements to build wheel


In [ ]:
import os, sys, json, time, platform, subprocess
from pathlib import Path

REPO_URL = "https://github.com/niksharma99/COMS6156FinalProject.git"
REPO_NAME = "COMS6156FinalProject"
# TODO: switch to "main" once the eval/demo branch is merged.
REPO_BRANCH = "add-evaluation-dataset"

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    clone_target = Path("/content") / REPO_NAME
    if not clone_target.exists():
        print(f"Colab detected — cloning {REPO_URL}")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(clone_target)], check=True)
    os.chdir(clone_target)
    REPO_ROOT = clone_target
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "dataset").exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "dataset").exists():
        raise RuntimeError(
            f"Could not find repo root (no 'dataset/' dir found walking up from {Path.cwd()})."
        )

SRC_DIR = str(REPO_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
except ImportError:
    print("python-dotenv not installed; relying on shell environment for API keys.")

from eval.sampling import sample_from_dataset
from eval.metrics import grade_mc, cosine_similarity, judge_with_claude, extract_mc_letter

print("Repo root:", REPO_ROOT)
print("Python:", platform.python_version())
print("In Colab:", IN_COLAB)
print("ANTHROPIC_API_KEY loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))


## 2. Sample 5 questions from each dataset

In [2]:
N_PER_DATASET = 5
DATASETS = ["standard_questions", "open_ended_hard", "reddit_questions"]

sampled = []
for ds in DATASETS:
    sampled.extend(sample_from_dataset(ds, N_PER_DATASET, seed=7))

print(f"Total items: {len(sampled)}")
for r in sampled:
    kind = "MC" if r.get("type") == "multiple_choice" else "OE"
    print(f"  [{kind}] [{r['_dataset']:<18} / {r['_topic']:<15}] {r['id']:<12} {r['question'][:70]}")

Total items: 15
  [OE] [standard_questions / budgeting      ] bud-003      What is the difference between a need and a want in the context of bud
  [MC] [standard_questions / credit_and_debt] crd-003      Which factor has the largest impact on your FICO credit score?
  [MC] [standard_questions / investing      ] inv-004      What does SIPC (Securities Investor Protection Corporation) protect ag
  [OE] [standard_questions / retirement     ] ret-005      What is the difference between a traditional IRA and a Roth IRA in ter
  [MC] [standard_questions / tax            ] tax-002      What is the capital gains tax rate for someone in the 32% income tax b
  [OE] [open_ended_hard    / budgeting      ] bud-008      If someone gets paid irregularly as a freelancer, is the standard advi
  [OE] [open_ended_hard    / credit_and_debt] cd-004       A borrower consolidates several credit card balances into a personal l
  [OE] [open_ended_hard    / investing      ] inv-004      Why can diversification

## 3. Load the baseline model

Qwen2.5-3B-Instruct runs locally on CPU / Apple Silicon (MPS) / CUDA. First load will download ~6 GB of weights to the HF cache.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
# bfloat16 on MPS avoids fp16-triggered CPU fallbacks that make generation crawl.
# On CUDA, fp16 is fine. On CPU, fp32 is the only option.
if device == "cuda":
    dtype = torch.float16
elif device == "mps":
    dtype = torch.bfloat16
else:
    dtype = torch.float32

# Let unsupported MPS ops fall back to CPU instead of failing (surfaces slow ops instead of hanging).
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

print("Device:", device, "| dtype:", dtype)

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"Tokenizer loaded in {time.time()-t0:.1f}s")

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device)
model.eval()
print(f"Model loaded + moved to {device} in {time.time()-t0:.1f}s")

# Warmup: first MPS/CUDA call compiles kernels and is much slower than steady-state.
print("Warming up with a 5-token dummy generation...", flush=True)
t0 = time.time()
_warm = tok("Hello", return_tensors="pt").to(device)
with torch.no_grad():
    model.generate(**_warm, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
print(f"Warmup done in {time.time()-t0:.1f}s")

Device: mps | dtype: torch.bfloat16


Tokenizer loaded in 1.2s


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded + moved to mps in 21.7s
Warming up with a 5-token dummy generation...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## 4. Prompt templates

For MC items we ask for a single letter so grading is mechanical. For open-ended items we ask for a concise, structured answer.

In [ ]:
from transformers import TextStreamer

SYSTEM_PROMPT = (
    "You are a careful personal-finance assistant. "
    "Give direct, accurate, and nuanced answers. "
    "When tradeoffs exist, acknowledge them instead of giving one-sided advice."
)

# Adaptive token budgets: MC items need ~1 letter; OE ~300 is plenty for 4-8 sentences.
MAX_NEW_TOKENS_MC = 10
MAX_NEW_TOKENS_OE = 300


def build_prompt(item: dict) -> list[dict]:
    if item.get("type") == "multiple_choice":
        options = "\n".join(item["options"])
        user = (
            f"{item['question']}\n\n{options}\n\n"
            "Respond with ONLY the single letter (A, B, C, or D) of the best answer."
        )
    else:
        user = (
            f"{item['question']}\n\n"
            "Answer in 4-8 sentences. Be specific and mention tradeoffs where relevant."
        )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]


def generate(item: dict, max_new_tokens: int | None = None, stream: bool = True, verbose: bool = True) -> str:
    """Generate an answer with optional live token streaming and timing logs."""
    if max_new_tokens is None:
        max_new_tokens = MAX_NEW_TOKENS_MC if item.get("type") == "multiple_choice" else MAX_NEW_TOKENS_OE

    messages = build_prompt(item)

    t_tok = time.time()
    enc = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)
    prompt_len = input_ids.shape[-1]
    if verbose:
        print(f"    [tokenize] {time.time()-t_tok:.2f}s  prompt_len={prompt_len}  max_new={max_new_tokens}", flush=True)

    streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True) if stream else None

    t_gen = time.time()
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
            streamer=streamer,
        )
    gen_tokens = out.shape[-1] - prompt_len
    dt = time.time() - t_gen
    if verbose:
        tps = gen_tokens / dt if dt > 0 else 0.0
        print(f"\n    [generate] {dt:.2f}s  new_tokens={gen_tokens}  ({tps:.1f} tok/s)", flush=True)

    gen = out[0, prompt_len:]
    return tok.decode(gen, skip_special_tokens=True).strip()

## 5. Run the baseline on all 15 items

In [ ]:
# Set RUN_ITEMS to a slice for quick testing, or `sampled` for the full run.
RUN_ITEMS = sampled[:1]  # change to `sampled` once you're happy with the first one

results = []
total = len(RUN_ITEMS)
print(f"Running baseline on {total} item(s)\n", flush=True)

for i, item in enumerate(RUN_ITEMS, 1):
    print(f"[{i}/{total}] {item['_dataset']}/{item['_topic']}/{item['id']}  ({item.get('type', 'open_ended')})", flush=True)
    print(f"  Q: {item['question'][:120]}{'…' if len(item['question']) > 120 else ''}", flush=True)
    t0 = time.time()
    answer = generate(item, stream=True, verbose=True)
    dt = time.time() - t0
    results.append({
        "id": item["id"],
        "dataset": item["_dataset"],
        "topic": item["_topic"],
        "type": item.get("type"),
        "difficulty": item.get("difficulty"),
        "question": item["question"],
        "reference": item["correct_answer"],
        "candidate": answer,
        "latency_sec": round(dt, 2),
    })
    print(f"  → total {dt:.1f}s  answer[:100]: {answer[:100]}\n", flush=True)

print(f"Done. {len(results)} items in {sum(r['latency_sec'] for r in results):.1f}s total.")

## 6. Score

### 6a. Multiple-choice accuracy

In [ ]:
mc_rows = [r for r in results if r["type"] == "multiple_choice"]
for r in mc_rows:
    g = grade_mc(r["candidate"], r["reference"])
    r["mc_correct"] = g["is_correct"]
    r["mc_picked"] = g["picked"]

if mc_rows:
    correct = sum(1 for r in mc_rows if r["mc_correct"])
    print(f"MC accuracy: {correct}/{len(mc_rows)} = {correct/len(mc_rows):.1%}")
else:
    print("No MC items in this sample.")

### 6b. Embedding similarity (open-ended)

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
oe_rows = [r for r in results if r["type"] != "multiple_choice"]

for r in oe_rows:
    emb_ref, emb_cand = embedder.encode([r["reference"], r["candidate"]], normalize_embeddings=True)
    r["embed_cosine"] = cosine_similarity(emb_ref, emb_cand)

if oe_rows:
    import statistics
    print(f"Mean cosine vs reference: {statistics.mean(r['embed_cosine'] for r in oe_rows):.3f}")

### 6c. LLM-as-judge (open-ended)

Requires `ANTHROPIC_API_KEY`. If missing, this cell will be skipped.

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic
    judge_client = anthropic.Anthropic()
    for r in oe_rows:
        score = judge_with_claude(
            question=r["question"],
            reference=r["reference"],
            candidate=r["candidate"],
            client=judge_client,
        )
        r["judge"] = score.to_dict()
        print(f"{r['id']:<12}  f={score.factuality} c={score.completeness} a={score.advice_quality}  mean={score.mean:.2f}")
else:
    print("ANTHROPIC_API_KEY not set — skipping LLM judge. Set it and re-run this cell.")

## 7. Summary tables

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df.head()

In [ ]:
# Per-dataset aggregates
def agg(g):
    row = {"n": len(g)}
    if "mc_correct" in g.columns:
        mc = g.dropna(subset=["mc_correct"])
        row["mc_accuracy"] = mc["mc_correct"].mean() if len(mc) else None
    if "embed_cosine" in g.columns:
        row["mean_cosine"] = g["embed_cosine"].mean()
    if "judge" in g.columns:
        judged = g.dropna(subset=["judge"])
        if len(judged):
            row["judge_factuality"] = judged["judge"].apply(lambda j: j["factuality"]).mean()
            row["judge_completeness"] = judged["judge"].apply(lambda j: j["completeness"]).mean()
            row["judge_advice"] = judged["judge"].apply(lambda j: j["advice_quality"]).mean()
            row["judge_mean"] = judged["judge"].apply(lambda j: j["mean"]).mean()
    row["mean_latency_sec"] = g["latency_sec"].mean()
    return pd.Series(row)

df.groupby("dataset").apply(agg)

## 8. Persist results

Saves a JSON file under `src/notebooks/results/` so the RAG notebook can load it later for side-by-side comparison.

In [ ]:
OUT_DIR = REPO_ROOT / "src" / "notebooks" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / f"baseline_{MODEL_ID.split('/')[-1]}.json"
out_path.write_text(json.dumps(results, indent=2))
print("Wrote", out_path)

## 9. (Optional) Claude-without-RAG second baseline

This isolates the contribution of retrieval. Uncomment to run once you're happy with the Qwen results.

In [ ]:
# if os.environ.get("ANTHROPIC_API_KEY"):
#     import anthropic
#     client = anthropic.Anthropic()
#     CLAUDE_MODEL = "claude-sonnet-4-6"
#     claude_results = []
#     for item in sampled:
#         msgs = build_prompt(item)
#         resp = client.messages.create(
#             model=CLAUDE_MODEL, max_tokens=600,
#             system=msgs[0]["content"],
#             messages=[{"role": "user", "content": msgs[1]["content"]}],
#         )
#         claude_results.append({
#             "id": item["id"], "dataset": item["_dataset"], "topic": item["_topic"],
#             "type": item.get("type"), "question": item["question"],
#             "reference": item["correct_answer"], "candidate": resp.content[0].text.strip(),
#         })
#     (OUT_DIR / f"baseline_{CLAUDE_MODEL}_no_rag.json").write_text(json.dumps(claude_results, indent=2))
#     print("Saved Claude no-RAG baseline.")